# Optimisation

This is the Adam version of [the calibration notebook](05-Calibration.ipynb). It maximises the same log-likelihood, on the same parameter set. Every name in `PARAM_BOUNDS` moves. Everything else in `BASE_PARAMS` stays fixed. The contact matrix is the folded spline, and the dynamical model is the corrected default (same as the calibration notebook).

The loss is the negative of that log-likelihood. A uniform prior is flat on its interval, so the mode of the likelihood and the mode of the posterior are the same point. The bounds are applied the way the uniform prior applies them: each parameter is a sigmoid of an unconstrained coordinate, scaled onto its `[low, high]`. Adam steps in that coordinate, and a step cannot leave the open interval.

The start is `get_nuts_init_values()`, the same interior point NUTS uses. One gradient of this likelihood is about 0.3 seconds after compilation, so 100 steps is about a minute. Use the **tb-macro-summer4 (pixi)** kernel.


## Likelihood

Seven terms, the same error models, and the same solver as the calibration notebook: Dopri5, `rtol = atol = 1e-5`, `dtmax = 1`, `max_steps = 4000`. A solve that does not succeed contributes `-1e10`, and the cotangent of that failure is zero, so Adam does not step on a failed draw.


In [ ]:
import json
from datetime import UTC, datetime
from pathlib import Path

import jax
import optax
import plotly.graph_objects as go
from jax import jit, nn, value_and_grad
from jax import numpy as jnp

from tb_macro.calibration import (
    TERM_NAMES,
    build_mixing_cases,
    compile_case,
    make_log_likelihood,
)
from tb_macro.parameters import BASE_PARAMS, PARAM_BOUNDS, get_nuts_init_values

case = next(item for item in build_mixing_cases() if item.name == "spline")
compiled = compile_case(case)
evaluate = make_log_likelihood(compiled, mixing=case.mixing, fold=case.fold)
print(case.name)


## Bounds

The unconstrained coordinate is the logit of where the parameter sits between its bounds. At one half of the interval the coordinate is 0. The calibration prior is uniform on the same interval, so this is the coordinate NUTS moves in.


In [ ]:
NAMES = tuple(PARAM_BOUNDS)
LOW = jnp.asarray([PARAM_BOUNDS[name][0] for name in NAMES])
HIGH = jnp.asarray([PARAM_BOUNDS[name][1] for name in NAMES])
START = get_nuts_init_values()


def constrain(raw: jax.Array) -> jax.Array:
    """Map an unconstrained coordinate onto the open prior interval."""
    return LOW + (HIGH - LOW) * nn.sigmoid(raw)


def unconstrain(value: jax.Array) -> jax.Array:
    """Inverse of ``constrain``. The start is interior, so this is finite."""
    unit = (value - LOW) / (HIGH - LOW)
    return jnp.log(unit) - jnp.log1p(-unit)


def unpack(raw: jax.Array) -> dict[str, jax.Array]:
    params = dict(BASE_PARAMS)
    for name, value in zip(NAMES, constrain(raw), strict=True):
        params[name] = value
    return params


raw = unconstrain(jnp.asarray([START[name] for name in NAMES]))
for name, (low, high) in PARAM_BOUNDS.items():
    print(f"{name:28s}  start {START[name]:.6g}  Uniform({low}, {high})")


## Adam

One hundred steps at learning rate `0.01` on the unconstrained coordinate. The printed number is the negative log-likelihood, so it should fall. The first step pays for compilation.


In [ ]:
N_STEPS = 100
LEARNING_RATE = 0.01


def loss(point: jax.Array) -> jax.Array:
    """Negative calibration log-likelihood."""
    return -evaluate(unpack(point))["total"]


optimizer = optax.adam(LEARNING_RATE)
opt_state = optimizer.init(raw)


@jit
def step(point: jax.Array, state: optax.OptState) -> tuple[jax.Array, optax.OptState, jax.Array]:
    value, grad = value_and_grad(loss)(point)
    updates, state = optimizer.update(grad, state, point)
    point = optax.apply_updates(point, updates)
    return point, state, value


history: list[float] = []
for i in range(N_STEPS):
    raw, opt_state, value = step(raw, opt_state)
    history.append(float(value))
    if i % 10 == 0:
        print(f"step {i:3d}  nll {history[-1]:.4f}", flush=True)
history.append(float(loss(raw)))
print(f"step {N_STEPS:3d}  nll {history[-1]:.4f}", flush=True)


## Fit

The curve is the loss Adam saw. The table is the start, the fitted point, and the prior interval. The fitted point is written under `outputs/local`, which is gitignored.


In [ ]:
figure = go.Figure(
    go.Scatter(x=list(range(len(history))), y=history, mode="lines")
)
figure.update_layout(
    xaxis_title="step",
    yaxis_title="negative log-likelihood",
    template="plotly_white",
)
figure.show()

fitted = {name: float(value) for name, value in zip(NAMES, constrain(raw), strict=True)}
final = evaluate(unpack(raw))
print(f"log-likelihood {float(final['total']):.4f}  ok {bool(final['ok'])}")
terms = "  ".join(f"{name} {float(final[name]):.3f}" for name in TERM_NAMES)
print(terms)
for name, (low, high) in PARAM_BOUNDS.items():
    print(f"{name:28s}  {START[name]:.6g} -> {fitted[name]:.6g}  [{low}, {high}]")

root = Path.cwd()
if not (root / "notebooks").is_dir():
    root = root.parent
out_dir = root / "outputs" / "local"
out_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
path = out_dir / f"optimise-{timestamp}.json"
path.write_text(
    json.dumps(
        {
            "log_likelihood": float(final["total"]),
            "negative_log_likelihood": history,
            "start": {name: START[name] for name in NAMES},
            "fitted": fitted,
            "terms": {name: float(final[name]) for name in TERM_NAMES},
        },
        indent=2,
    )
    + "\n"
)
print(path)
